# Market Data Automation — Exploratory Data Analysis

This notebook provides an end-to-end walkthrough of the market data pipeline:
1. Loading processed data
2. Data quality checks
3. Missing value analysis
4. Feature distribution analysis
5. Cross-source comparison
6. Risk metric visualisation

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Project modules
from utils.config import SYMBOLS, PROC_DATA_DIR, START_DATE, END_DATE
from utils.logger import get_logger

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid')

log = get_logger('notebook')
print('✅ Environment ready.')
print(f'Symbols   : {SYMBOLS}')
print(f'Date range: {START_DATE} → {END_DATE}')

## 1. Load Processed Data

In [ ]:
def load_features(symbol):
    safe = symbol.replace('.', '_')
    path = os.path.join(PROC_DATA_DIR, f'features_{safe}.csv')
    if not os.path.exists(path):
        print(f'⚠️  Features file not found: {path}')
        return pd.DataFrame()
    return pd.read_csv(path, parse_dates=['date'])

data = {sym: load_features(sym) for sym in SYMBOLS}

for sym, df in data.items():
    if not df.empty:
        print(f'{sym}: {len(df)} rows, {df.shape[1]} columns, {df["date"].min().date()} → {df["date"].max().date()}')
    else:
        print(f'{sym}: ⚠️  No data (run pipeline first)')

## 2. Data Quality Checks

In [ ]:
# Missing value summary
print('=== Missing Value Summary ===')
for sym, df in data.items():
    if df.empty:
        continue
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f'{sym}: ✅ No missing values')
    else:
        print(f'\n{sym}:')
        print(missing.to_string())

In [ ]:
# Missing value heatmap (first available symbol)
available = [(sym, df) for sym, df in data.items() if not df.empty]
if available:
    sym, df = available[0]
    fig, ax = plt.subplots(figsize=(14, 4))
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    sns.heatmap(df[numeric_cols].isna().T, cbar=False, ax=ax, cmap='YlOrRd')
    ax.set_title(f'Missing Value Heatmap — {sym}')
    ax.set_xlabel('Time')
    plt.tight_layout()
    plt.show()

## 3. Price & Volume History

In [ ]:
fig, axes = plt.subplots(len(available), 2, figsize=(16, 4 * len(available)))
if len(available) == 1:
    axes = [axes]

for i, (sym, df) in enumerate(available):
    ax1, ax2 = axes[i]
    
    # Close price
    ax1.plot(df['date'], df['close'], linewidth=1.2, color='steelblue')
    ax1.set_title(f'{sym} — Close Price')
    ax1.set_ylabel('Price (INR)')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    # Volume
    if 'volume' in df.columns:
        ax2.bar(df['date'], df['volume'], color='orange', alpha=0.6, width=1)
        ax2.set_title(f'{sym} — Volume')
        ax2.set_ylabel('Volume')
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

## 4. Feature Distribution Analysis

In [ ]:
features = ['log_return', 'rolling_volatility_20', 'momentum_14', 'rsi_14', 'volume_zscore', 'beta_rolling']

for sym, df in available:
    available_feats = [f for f in features if f in df.columns]
    if not available_feats:
        continue
    
    n = len(available_feats)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    
    for ax, feat in zip(axes, available_feats):
        series = df[feat].dropna()
        ax.hist(series, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(series.mean(), color='red', linestyle='--', label=f'Mean={series.mean():.3f}')
        ax.set_title(feat.replace('_', ' ').title())
        ax.legend(fontsize=8)
    
    fig.suptitle(f'Feature Distributions — {sym}', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Rolling Volatility & RSI

In [ ]:
for sym, df in available:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Close price
    ax1.plot(df['date'], df['close'], color='steelblue', linewidth=1)
    ax1.set_ylabel('Close')
    ax1.set_title(f'{sym} — Technical Overview')
    
    # Volatility
    if 'rolling_volatility_20' in df.columns:
        ax2.plot(df['date'], df['rolling_volatility_20'], color='darkorange', linewidth=1)
        ax2.set_ylabel('Annualised Vol')
        ax2.axhline(df['rolling_volatility_20'].mean(), color='red', linestyle='--', alpha=0.5, label='Mean')
        ax2.legend()
    
    # RSI
    if 'rsi_14' in df.columns:
        ax3.plot(df['date'], df['rsi_14'], color='purple', linewidth=1)
        ax3.axhline(70, color='red',   linestyle='--', alpha=0.5, label='Overbought (70)')
        ax3.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
        ax3.set_ylabel('RSI(14)')
        ax3.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Risk Metrics Summary

In [ ]:
import glob

risk_reports = []
for path in glob.glob(os.path.join(PROC_DATA_DIR, 'risk_report_*.csv')):
    try:
        risk_reports.append(pd.read_csv(path))
    except Exception:
        pass

if risk_reports:
    risk_df = pd.concat(risk_reports, ignore_index=True)
    display_cols = [
        'symbol', 'sharpe_ratio', 'sortino_ratio',
        'max_drawdown_pct', 'annualised_volatility',
        'var_95_historical', 'cvar_95', 'calmar_ratio'
    ]
    display_cols = [c for c in display_cols if c in risk_df.columns]
    print(risk_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))
else:
    print('⚠️  No risk report files found. Run: python models/risk_model.py')

## 7. Correlation Matrix

In [ ]:
# Build a returns correlation matrix across all symbols
returns_dict = {}
for sym, df in available:
    if 'log_return' in df.columns:
        s = df.set_index('date')['log_return']
        returns_dict[sym] = s

if len(returns_dict) > 1:
    returns_df  = pd.DataFrame(returns_dict).dropna(how='all')
    corr_matrix = returns_df.corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='coolwarm', center=0, square=True, ax=ax,
        linewidths=0.5
    )
    ax.set_title('Log Return Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Need ≥2 symbols with data for correlation analysis.')